# Map plots

Produces the two trajectory-grid figures used in the paper (`figures/trajectory_grid.pdf`,
`figures/busy_trajectory_grid.pdf`): NEREUS_best (`nereus_ablation/version_15`) predictions
overlaid on the region map for a hand-picked, Pareto-diverse set of curved trajectories, and
for a set of trajectories with many nearby vessels.

Positions live in `METER_CRS` (EPSG:25832) -- `cords_to_meters` converts them during dataset
construction -- so the `land.geojson` background is reprojected to the same CRS before
plotting. Requires the raw AIS dataset (not included in this repo, see the README's Data
section) and a GPU (model inference).

This is a trimmed-down version of the exploratory notebook used during development --
only the cells that produced the final figures are kept.

In [ ]:
import sys
sys.path.insert(0, "src")  # .env sets PYTHONPATH=src, but the kernel may not load it

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
from pathlib import Path

import matplotlib.colors as mcolors
import torch
from matplotlib import cm
from torch.utils.data import Subset
from torch_geometric.loader import DataLoader
from tqdm import tqdm

from data.graph.build_dataloader import graph_loader
from data.map.rasterize import Rasterizer
from data.map.scene_gernerator import SceneLoader
from train.pl_modules import NereusModule
from utils.config import DATA_FOLDER_PATH, METER_CRS, STEPS_PER_MINUTE, STEP_SIZE

SOURCE = "fh"
REGION = "kiel"
DE_NORMALIZE = 100  # normalized rel. displacement -> meters (matches full_eval_nereus.py)

# Region bounding boxes (lon_min, lat_min, lon_max, lat_max), from full_eval_nereus.py.
ALL_REGIONS = {
    "kiel":        [10.12, 54.31, 10.33, 54.46],
    "aarhus":      [10.21, 56.04, 10.47, 56.17],
    "odense":      [10.42, 55.42, 10.68, 55.55],
    "little_belt": [9.64,  55.25,  9.90, 55.37],
}

In [ ]:
def load_land(region=REGION, source=SOURCE):
    """Region land polygons reprojected to METER_CRS (same frame as the trajectories)."""
    path = DATA_FOLDER_PATH / f"maps/2_standardized/{source}_10/{region}/land.geojson"
    return gpd.read_file(path).to_crs(METER_CRS)


def swap_rasterizer(model, bbox):
    """Point the model (and its map/prior CNNs) at this region's rasterizer."""
    rasterizer = Rasterizer(bbox)
    model.rasterizer = rasterizer
    if hasattr(model, "map_cnn") and model.map_cnn is not None:
        model.map_cnn.rasterizer = rasterizer
    if hasattr(model, "prior_cnn") and model.prior_cnn is not None:
        model.prior_cnn.rasterizer = rasterizer
    return rasterizer


def load_model(ckpt_dir, bbox, region, device, source="fh"):
    """Load a Lightning NereusModule checkpoint and build the region scene tensor."""
    ckpt_path = sorted(Path(ckpt_dir).rglob("*.ckpt"))[0]  # e.g. version_15/best.ckpt
    pl_module = NereusModule.load_from_checkpoint(str(ckpt_path), map_location=device)
    model = pl_module.model.to(device)
    model.eval()
    swap_rasterizer(model, bbox)

    map_folder = DATA_FOLDER_PATH / f"maps/2_standardized/{source}_10/{region}"
    scene = torch.from_numpy(
        np.ascontiguousarray(SceneLoader(Rasterizer(bbox)).load_scene(map_folder))
    ).to(device, torch.float32)
    return pl_module, model, scene

In [ ]:
def gt_geometry_batch(y_pos, y_mask):
    """Vectorized gt_geometry over a batch.

    y_pos [G, T, 2], y_mask [G, T] bool (a contiguous prefix per row).
    Returns a dict of [G] tensors. Padding steps are (0, 0), so everything is masked.
    """
    p = y_pos.double()
    m = y_mask.bool()
    G = p.shape[0]
    idx = torch.arange(G, device=p.device)
    counts = m.sum(1)                                    # valid length per row

    seg = p[:, 1:] - p[:, :-1]                           # [G, T-1, 2]
    seg_m = m[:, 1:] & m[:, :-1]                          # segment valid iff both ends valid
    path_len = (seg.norm(dim=-1) * seg_m).sum(1)         # [G]

    first = p[:, 0]                                      # prefix mask -> index 0 is valid when counts>0
    last = p[idx, (counts - 1).clamp(min=0)]             # last valid position
    chord = (last - first).norm(dim=-1)

    # total absolute turning angle (wrapped per step)
    ang = torch.atan2(seg[..., 1], seg[..., 0])          # [G, T-1]
    dang = (ang[:, 1:] - ang[:, :-1] + math.pi) % (2 * math.pi) - math.pi
    dang_m = seg_m[:, 1:] & seg_m[:, :-1]
    turn = (dang.abs() * dang_m).sum(1)                  # radians

    # max perpendicular deviation from the start->end chord
    d = (last - first) / chord.clamp(min=1e-6).unsqueeze(-1)     # [G, 2] unit chord dir
    rel = p - first.unsqueeze(1)                                 # [G, T, 2]
    perp = rel - (rel * d.unsqueeze(1)).sum(-1, keepdim=True) * d.unsqueeze(1)
    max_dev = perp.norm(dim=-1).masked_fill(~m, 0.0).max(dim=1).values

    sinuosity = torch.where(chord > 1e-6, path_len / chord.clamp(min=1e-6),
                            torch.ones_like(chord))
    turn = turn.masked_fill(counts < 3, 0.0)             # need >=2 segments to turn
    max_dev = max_dev.masked_fill(counts < 2, 0.0)

    return {
        "n": counts, "path_len": path_len, "chord": chord,
        "sinuosity": sinuosity, "turn_deg": torch.rad2deg(turn), "max_dev": max_dev,
    }


def _resolve_base(dset):
    """Unwrap (possibly nested) torch Subsets -> (base_dataset, map) where
    map[pos] is the base-dataset index of the pos-th sample of `dset`."""
    idx = list(range(len(dset)))
    base = dset
    while isinstance(base, Subset):
        idx = [base.indices[j] for j in idx]
        base = base.dataset
    return base, idx


def geometry_dataframe(dset, batch_size=512, num_workers=4, device=None):
    """Iterate the dataset once (shuffle=False, drop_last=False) and return a DataFrame of
    per-sample GT geometry. Accepts a full dataset or a torch Subset.

    `index` is the position in the passed `dset` (so `dset[index]` works); `base_index`
    is the index into the underlying dataset (for the full dataset)."""
    loader = DataLoader(dset, batch_size=batch_size, shuffle=False, drop_last=False,
                        num_workers=num_workers)
    device = device or torch.device("cpu")
    frames, start = [], 0
    for batch in tqdm(loader, desc="gt_geometry"):
        geo = gt_geometry_batch(batch.y_pos.to(device), batch.y_mask.to(device))
        df = pd.DataFrame({k: v.cpu().numpy() for k, v in geo.items()})
        df.insert(0, "index", np.arange(start, start + len(df)))
        frames.append(df)
        start += len(df)
    df = pd.concat(frames, ignore_index=True)

    base, base_map = _resolve_base(dset)  # handles Subset (or plain dataset -> identity map)
    df.insert(1, "base_index", [base_map[i] for i in df["index"]])
    df["target_id"] = [base.items[b][1] for b in df["base_index"]]  # avoids relying on PyG collation
    return df


def _fit_metrics(pred_abs, pred_abs_k, y_pos, y_mask):
    """Per-sample displacement metrics for a batch. Shapes: pred_abs [G,T,2],
    pred_abs_k [G,K,T,2], y_pos [G,T,2], y_mask [G,T]. Returns dict of [G] tensors."""
    G, K = pred_abs_k.shape[0], pred_abs_k.shape[1]
    m = y_mask.float()
    counts = y_mask.sum(1)
    last = (counts - 1).clamp(min=0)
    gi = torch.arange(G, device=pred_abs.device)

    de = (pred_abs - y_pos).norm(dim=-1)                          # [G,T]
    ade = (de * m).sum(1) / m.sum(1).clamp(min=1)
    fde = de[gi, last]                                            # error at last valid step
    de_k = (pred_abs_k - y_pos.unsqueeze(1)).norm(dim=-1)         # [G,K,T]
    ade_k = (de_k * m.unsqueeze(1)).sum(2) / m.sum(1, keepdim=True).clamp(min=1)   # [G,K]
    fde_k = de_k.gather(2, last.view(G, 1, 1).expand(G, K, 1)).squeeze(2)          # [G,K]
    return dict(ade=ade, fde=fde, min_ade_k=ade_k.min(1).values,
                min_fde_k=fde_k.min(1).values, n=counts)


def predict_dataframe(subset, pl_module, model, scene, device,
                      batch_size=256, num_workers=4, store_pred=True):
    """Run `model` over a subset (batched) and return predictions + fit metrics per sample.

    Same keys as geometry_dataframe (`index`/`base_index`/`target_id`) so the two merge.
    Metric columns: ade, fde, min_ade_k, min_fde_k (meters), n (valid horizon length).
    With store_pred, adds object columns pred_abs [T,2], pred_abs_k [K,T,2], pi_k [K]."""
    cfg = pl_module.cfg
    T, K = cfg.pred_len, cfg.mdn_modes
    loader = DataLoader(subset, batch_size=batch_size, shuffle=False, drop_last=False,
                        num_workers=num_workers)
    base, base_map = _resolve_base(subset)

    model.eval()
    frames, start = [], 0
    with torch.inference_mode():
        for batch in tqdm(loader, desc="predict"):
            batch = batch.to(device)
            ego_idx = batch.is_ego.nonzero(as_tuple=True)[0]
            G = ego_idx.numel()

            mdn_out = model(batch, scene).view(G, T, K, 5)
            pi = torch.softmax(mdn_out[..., 0], dim=-1)          # [G,T,K]
            mu = mdn_out[..., 1:3]                                # [G,T,K,2]
            last = batch.x_pos[ego_idx, -1:, :]                  # [G,1,2]
            exp_rel = (pi.unsqueeze(-1) * mu).sum(2)             # [G,T,2]
            pred_abs = torch.cumsum(exp_rel, 1) * DE_NORMALIZE + last
            mu_k = mu.permute(0, 2, 1, 3)                        # [G,K,T,2]
            pred_abs_k = torch.cumsum(mu_k, 2) * DE_NORMALIZE + last.unsqueeze(1)
            pi_k = pi.mean(1)                                    # [G,K]

            metrics = _fit_metrics(pred_abs, pred_abs_k, batch.y_pos, batch.y_mask)
            df = pd.DataFrame({k: v.cpu().numpy() for k, v in metrics.items()})
            df.insert(0, "index", np.arange(start, start + G))
            if store_pred:
                df["pred_abs"] = list(pred_abs.cpu().numpy().astype("float32"))
                df["pred_abs_k"] = list(pred_abs_k.cpu().numpy().astype("float32"))
                df["pi_k"] = list(pi_k.cpu().numpy().astype("float32"))
            frames.append(df)
            start += G

    out = pd.concat(frames, ignore_index=True)
    out.insert(1, "base_index", [base_map[i] for i in out["index"]])
    out["target_id"] = [base.items[b][1] for b in out["base_index"]]
    return out


def pareto_front(df, x="ade", y="turn_deg", minimize_x=True, maximize_y=True):
    """Non-dominated rows for two objectives. Default: minimize x (ade), maximize y (turn_deg).

    A row is Pareto-optimal if no other row is at least as good in both objectives and
    strictly better in at least one. O(n log n) skyline sweep. Non-finite rows are dropped.
    Returns the optimal rows sorted along x.
    """
    sub = df[np.isfinite(df[x]) & np.isfinite(df[y])]
    sx = sub[x].to_numpy(float) * (1 if minimize_x else -1)
    sy = sub[y].to_numpy(float) * (1 if maximize_y else -1)
    order = np.lexsort((-sy, sx))  # x ascending, ties broken by y descending
    keep, best_y = [], -np.inf
    for i in order:
        if sy[i] > best_y:          # nothing with smaller/equal x has a higher y -> optimal
            keep.append(i)
            best_y = sy[i]
    return sub.iloc[keep].sort_values(x, ascending=minimize_x)

In [ ]:
def _traj_datetime(dset, base_index):
    """Return the observation timestamp of a dataset sample as a pd.Timestamp."""
    cur_t, _ = dset.items[base_index]
    return pd.Timestamp(cur_t * STEP_SIZE, unit="s")


def draw_traj_ax(ax, data, land, pred, title=None, pad=300, show_legend=False):
    """Draw one trajectory (+ MDN prediction) onto an existing Axes."""
    ego_obs = data.x_pos[0].numpy()[data.x_mask[0].numpy()]
    ego_fut = data.y_pos[0].numpy()[data.y_mask[0].numpy()]
    pred_abs   = pred["pred_abs"]    # [T, 2]
    pred_abs_k = pred["pred_abs_k"]  # [K, T, 2]
    pi_k       = pred["pi_k"]        # [K]

    land.plot(ax=ax, facecolor="lightgray", edgecolor="black", alpha=0.5)

    for i in range(1, data.x_pos.shape[0]):
        nb = data.x_pos[i].numpy()[data.x_mask[i].numpy()]
        if len(nb):
            ax.plot(nb[:, 0], nb[:, 1], color="gray", lw=0.8, alpha=0.5,
                    label="Neighbours" if (show_legend and i == 1) else None)

    ax.scatter(ego_obs[:, 0], ego_obs[:, 1], color="blue", s=5, alpha=0.8,
               label="Observed" if show_legend else None)
    ax.scatter(ego_fut[:, 0], ego_fut[:, 1], color="green", s=5, alpha=0.8,
               label="Ground truth" if show_legend else None)

    cmap_m = cm.plasma
    norm_m = mcolors.Normalize(vmin=0.0, vmax=1.0)
    for k in range(pred_abs_k.shape[0]):
        ax.plot(pred_abs_k[k, :, 0], pred_abs_k[k, :, 1],
                color=cmap_m(norm_m(pi_k[k])), lw=1.0, alpha=0.85, zorder=2,
                label=f"Mode {k} (π={pi_k[k]:.2f})" if show_legend else None)
    ax.plot(pred_abs[:, 0], pred_abs[:, 1], color="red", lw=1.5, ls="--", zorder=3,
            label="Expected" if show_legend else None)

    all_xy = np.concatenate([ego_obs, ego_fut, pred_abs, pred_abs_k.reshape(-1, 2)])
    (minx, miny), (maxx, maxy) = all_xy.min(0), all_xy.max(0)
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    half = max(maxx - minx, maxy - miny) / 2 + pad
    ax.set_xlim(cx - half, cx + half)
    ax.set_ylim(cy - half, cy + half)
    ax.set_aspect("equal", adjustable="box")
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax.grid(True, linestyle=":", linewidth=0.5, alpha=0.5)

    if title:
        ax.set_title(title, fontsize=7.5, pad=3)
    if show_legend:
        ax.legend(fontsize=6, loc="best", markerscale=0.8)


def plot_trajectory_grid(rows, dset, land, nrows=2, ncols=4,
                         index_col="base_index", figsize=None,
                         pad=300, hspace=0.15, suptitle=None):
    """nrows x ncols trajectory grid.

    Legend is placed in the top-right panel; a shared mode-probability
    colorbar is added to the right of the whole figure.
    """
    rows = rows.head(nrows * ncols)
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=figsize or (ncols * 4, nrows * 4), gridspec_kw={"hspace": hspace})
    axes = np.array(axes).reshape(nrows, ncols)

    for idx, (_, row) in enumerate(rows.iterrows()):
        r, c = divmod(idx, ncols)
        ax = axes[r, c]

        data = dset[int(row[index_col])]
        pred = {
            "pred_abs":   np.asarray(row["pred_abs"]),
            "pred_abs_k": np.asarray(row["pred_abs_k"]),
            "pi_k":       np.asarray(row["pi_k"]),
        }

        dt = _traj_datetime(dset, int(row[index_col]))
        title = dt.strftime("%Y-%m-%d  %H:%M UTC")

        # legend only on top-right panel
        draw_traj_ax(ax, data, land, pred, title=title, pad=pad,
                     show_legend=(r == 0 and c == ncols - 1))

    for idx in range(len(rows), nrows * ncols):
        r, c = divmod(idx, ncols)
        axes[r, c].set_visible(False)

    # shared colorbar (must come before suptitle)
    sm = cm.ScalarMappable(cmap=cm.plasma, norm=mcolors.Normalize(vmin=0.0, vmax=1.0))
    sm.set_array([])
    fig.colorbar(sm, ax=axes.ravel().tolist(),
                 label="Mode probability (π)", shrink=0.6, pad=0.01)

    if suptitle:
        visible = [ax for ax in axes.ravel() if ax.get_visible()]
        x0 = min(ax.get_position().x0 for ax in visible)
        x1 = max(ax.get_position().x1 for ax in visible)
        y1 = max(ax.get_position().y1 for ax in visible)
        fig.suptitle(suptitle, fontsize=12, ha="center",
                     x=(x0 + x1) / 2,   # centered over subplots, not full figure
                     y=y1 + 0.05)        # just above the top row

    return fig, axes

## Setup: land polygons, dataset, and the NEREUS_best checkpoint

In [ ]:
land = load_land()

# Load the whole dataset for the region (this reads the parquet files and builds the
# graph edges, so the first run takes a while; graph_loader caches it afterwards).
loader, dset = graph_loader(
    data_folder=DATA_FOLDER_PATH / f"ais/4_features/{SOURCE}_10/{REGION}",
    flag="test",
    min_date=pd.Timestamp("2022-01-01"),
    max_date=pd.Timestamp("2024-01-01"),
    batch_size=1,
    pred_len=30,
    obs_len=60,
    max_edge_dist=500,
    shuffle=False,
    ship_group="all",
)
print(f"samples: {len(dset)} | trajectories: {len(dset.pos_map)}")

device = torch.device("cuda:0")
bbox = ALL_REGIONS[REGION]
pl_module, model, scene = load_model(
    "lightning_logs/nereus_ablation/version_15", bbox, REGION, device, source=SOURCE
)

# Every dataset sample with at least one neighbour (x_pos includes the ego row, so >1).
neighbor_list = [(i, dset[i].x_pos.shape[0]) for i in tqdm(range(len(dset)), desc="scan neighbors")
                  if dset[i].x_pos.shape[0] > 1]
subset = Subset(dset, [x[0] for x in neighbor_list])

Path("figures").mkdir(exist_ok=True)

## Figure: `trajectory_grid.pdf`

Curved trajectories (>60° total turn), Pareto-diverse in (ADE, max deviation from chord).

In [ ]:
geom = geometry_dataframe(subset)
geom = geom[geom["n"] == 30]
geom = geom[geom["turn_deg"] > 60]

subset2 = Subset(subset, geom["index"].to_list())
df = predict_dataframe(subset2, pl_module, model, scene, device, batch_size=512)
df_pred = df.join(geom.set_index("base_index"), on="base_index", rsuffix="_geom")
df_pareto = pareto_front(df_pred, y="max_dev")

# Hand-picked from the Pareto-optimal set above (diverse turn angles / deviations).
df_chosen_traj = df_pareto[df_pareto["base_index"].isin(
    [2577241, 2048713, 1870201, 49147, 3386299, 3350627, 2367963, 370770, 1854238]
)]

fig, axes = plot_trajectory_grid(
    df_chosen_traj,
    dset=dset,
    land=land,
    nrows=3, ncols=3,
    pad=300,
    suptitle="Example trajectories",
    hspace=-0.3,
)
plt.show()
fig.savefig("figures/trajectory_grid.pdf", bbox_inches="tight")

## Figure: `busy_trajectory_grid.pdf`

Trajectories with many nearby vessels (>24 neighbours), one per target vessel.

In [ ]:
many_neighbors = [x[0] for x in filter(lambda x: x[1] > 10, neighbor_list)]
subset3 = Subset(dset, many_neighbors)
df_many = predict_dataframe(subset3, pl_module, model, scene, device, batch_size=512)

dt_list, num_neighbors = [], []
for i in many_neighbors:
    dt_list.append(_traj_datetime(dset, i))
    num_neighbors.append(dset[i].x_pos.shape[0])
df_many["time"] = dt_list
df_many["num_neighbors"] = num_neighbors
df_many["date"] = df_many["time"].dt.date

df_many = df_many[df_many["n"] == 30]

df_chosen = df_many[df_many["num_neighbors"] > 24]
df_chosen = df_chosen.drop_duplicates(subset="target_id", keep="first")

fig, axes = plot_trajectory_grid(
    df_chosen,
    dset=dset,
    land=land,
    nrows=2, ncols=3,
    pad=300,
    suptitle="Example trajectories with many neighbors",
    hspace=-0.3,
)
plt.show()
fig.savefig("figures/busy_trajectory_grid.pdf", bbox_inches="tight")